# mlmodels end-goal API, pillar 1 -- inject a NN + PRE-SPLIT loaders

**Goal (L3f.1).** Drive the `mlmodels` training seam with an injected NN **and**
pre-split train/val/test DataLoaders, where the **split is done by hand** in this
notebook (certified via the `make_holdout_spec` domain value object), run **per-epoch
validation**, and report **val + test** metrics on the model recovered from disk.

This is the notebook-first, hand-gluing workflow: discover the shape here, over real
neutron-star data with a simple model, then harvest whatever proves general into the
owning bounded context. Companion to `ns_mlmodels-e2e.ipynb` (the first integration
proof); this one adds the **split + validation** pillar.

**Headline finding (see the last cell):** injecting pre-split loaders needs **no `src`
change** -- the loaders ride in the injected adapter's closure, and the application port
`SaveTrainedRunFn(ConfiguredRun) -> Result[Checkpoint]` is agnostic to how many loaders
you inject.

## 0. Config / env seam

Same as the companion notebook: set `SURROGATE_MODELS__*` explicitly (walking up to
`pyproject.toml`) before importing the package, because `.env` is cwd-relative and the
defaults point at `/var` root.

In [ ]:
import os
from pathlib import Path


def project_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise RuntimeError("pyproject.toml not found above cwd")


ROOT = project_root()
VAR = ROOT / "var" / "data" / "surrogate_models"
CKPT_DIR = VAR / "checkpoints" / "nb-split"
os.environ["SURROGATE_MODELS__DATASETS__PATH"] = str(VAR / "datasets")
os.environ["SURROGATE_MODELS__DATASETS__NEUTRON_STARS_SOURCE"] = str(
    ROOT / "data" / "neutron-stars" / "neutron-stars.dat"
)
os.environ["SURROGATE_MODELS__MLMODELS__CHECKPOINT_DIR"] = str(CKPT_DIR)
os.environ["SURROGATE_MODELS__LOGGING__FILE"] = str(VAR / "log" / "surrogate_models.log")
print("checkpoint dir:", CKPT_DIR)

In [ ]:
%matplotlib inline
from functools import partial
from typing import Any

import lightning
import matplotlib.pyplot as plt
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from surrogate_models import load_neutron_stars
from surrogate_models.mlmodels.application import TrainRun, handle_train_run
from surrogate_models.mlmodels.domain import (
    Checkpoint,
    ConfiguredRun,
    make_holdout_spec,
    make_model_identity,
)
from surrogate_models.mlmodels.infrastructure import (
    RunManifest,
    load_trained_run,
    materialize_model,
    structural_fingerprint,
    write_run_manifest,
)
from surrogate_models.railway_adts import fmap_error, safe

torch.manual_seed(0)
_ = lightning.seed_everything(0, verbose=False)

## 1. Data + BY-HAND seeded split (certified via `make_holdout_spec`)

We load the real data, standardize two features -> one target, then split into
train/val/test **by hand** with a seeded permutation. The fractions + seed are certified
through the existing `make_holdout_spec` domain value object, so the split is recorded as
reproducible metadata even though the splitting itself is done here in the notebook.

In [ ]:
df = load_neutron_stars()
FEATURES, TARGET = ["rhoc", "Pc"], "M"
sample = df[[*FEATURES, TARGET]].dropna().sample(n=3000, random_state=0)
x_raw = torch.tensor(sample[FEATURES].to_numpy(), dtype=torch.float32)
y_raw = torch.tensor(sample[[TARGET]].to_numpy(), dtype=torch.float32)
x_mean, x_std = x_raw.mean(0), x_raw.std(0)
y_mean, y_std = y_raw.mean(0), y_raw.std(0)
X = (x_raw - x_mean) / x_std
Y = (y_raw - y_mean) / y_std

SEED = 0
SPLIT = make_holdout_spec(0.7, 0.15, 0.15, SEED).unwrap()  # certify split metadata
gen = torch.Generator().manual_seed(SEED)
perm = torch.randperm(X.shape[0], generator=gen)
n_train = int(SPLIT.train_fraction * X.shape[0])
n_val = int(SPLIT.val_fraction * X.shape[0])
idx_train, idx_val, idx_test = perm[:n_train], perm[n_train:n_train + n_val], perm[n_train + n_val:]


def make_loader(idx: torch.Tensor, *, shuffle: bool) -> DataLoader:
    return DataLoader(TensorDataset(X[idx], Y[idx]), batch_size=64, shuffle=shuffle)


train_loader = make_loader(idx_train, shuffle=True)
val_loader = make_loader(idx_val, shuffle=False)
test_loader = make_loader(idx_test, shuffle=False)
print(SPLIT)
print("sizes  train/val/test:", idx_train.numel(), idx_val.numel(), idx_test.numel())

## 2. A simple NN with a `validation_step`

Same tiny `2 -> 8 -> 1` MLP as before, now with a `validation_step` so Lightning runs
validation each epoch when a val loader is passed to `fit`.

In [ ]:
class SplitSurrogate(lightning.LightningModule):
    '''A 2->8->1 MLP with per-epoch validation -- notebook model only.'''

    MODEL_NAME = "ns-mlp-split"
    MODEL_VERSION = "0.1.0"

    def __init__(self, n_features: int, learning_rate: float, optimizer_name: str) -> None:
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_features, 8), nn.ReLU(), nn.Linear(8, 1))
        self.learning_rate = learning_rate
        self.optimizer_name = optimizer_name

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        out: torch.Tensor = self.net(features)
        return out

    def training_step(self, *args: Any, **kwargs: Any) -> torch.Tensor:
        features, targets = args[0]
        loss: torch.Tensor = nn.functional.mse_loss(self.net(features), targets)
        self.log("train_loss", loss, on_step=False, on_epoch=True, logger=False)
        return loss

    def validation_step(self, *args: Any, **kwargs: Any) -> torch.Tensor:
        features, targets = args[0]
        loss: torch.Tensor = nn.functional.mse_loss(self.net(features), targets)
        self.log("val_loss", loss, on_step=False, on_epoch=True, logger=False)
        return loss

    def configure_optimizers(self) -> torch.optim.Optimizer:
        match self.optimizer_name:
            case "adam":
                return torch.optim.Adam(self.parameters(), lr=self.learning_rate)
            case _:
                return torch.optim.SGD(self.parameters(), lr=self.learning_rate)


N_FEATURES = len(FEATURES)
IDENTITY = make_model_identity(SplitSurrogate.MODEL_NAME, SplitSurrogate.MODEL_VERSION).unwrap()


def build_model(config: Any) -> SplitSurrogate:
    '''Factory: build the surrogate FROM a run's certified config.'''
    return SplitSurrogate(N_FEATURES, config.learning_rate, config.optimizer)

## 3. Inject the NN + pre-split loaders into `handle_train_run`

The adapter closes over the **train and val** loaders (this is the injection point for
pre-split data) and fits with per-epoch validation. A `_Progress` callback captures the
train/val loss each epoch -- an early taste of the "continuous feedback" pillar (L3f.2).
The application port is unchanged: currying the loaders yields the same
`SaveTrainedRunFn(ConfiguredRun) -> Result[Checkpoint]` shape.

In [ ]:
class _Progress(lightning.Callback):
    '''Record per-epoch (epoch, train_loss, val_loss) -- continuous-feedback seed.'''

    def __init__(self) -> None:
        super().__init__()
        self.rows: list[tuple[int, float, float]] = []

    def on_validation_epoch_end(
        self, trainer: lightning.Trainer, pl_module: lightning.LightningModule
    ) -> None:
        m = trainer.callback_metrics
        if "train_loss" in m and "val_loss" in m:
            self.rows.append((trainer.current_epoch, float(m["train_loss"]), float(m["val_loss"])))


@safe(Exception, fmap_error(lambda cause: cause, code="TRAINING_FAILED"))
def save_with_validation(
    checkpoint_dir: Path,
    train_dl: DataLoader,
    val_dl: DataLoader,
    progress: _Progress,
    run: ConfiguredRun,
) -> Checkpoint:
    '''Fit with train+val loaders (per-epoch validation) and persist ckpt + manifest.'''
    model = build_model(run.config)
    trainer = lightning.Trainer(
        max_epochs=run.config.max_epochs, accelerator="cpu", devices=1, logger=False,
        enable_checkpointing=False, enable_progress_bar=False, enable_model_summary=False,
        callbacks=[progress],
    )
    trainer.fit(model, train_dl, val_dl)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    target = checkpoint_dir / f"{run.run_id}.ckpt"
    torch.save(model.state_dict(), target)
    write_run_manifest(checkpoint_dir, RunManifest(
        str(run.run_id), model.MODEL_NAME, model.MODEL_VERSION,
        run.config.max_epochs, run.config.learning_rate, run.config.batch_size,
        run.config.optimizer, structural_fingerprint(model.state_dict())))
    return Checkpoint(str(target))

## 4. Train, and watch validation per epoch

In [ ]:
progress = _Progress()
cmd = TrainRun("ns-split", max_epochs=20, learning_rate=0.05, batch_size=64, optimizer="adam")
save = partial(save_with_validation, CKPT_DIR, train_loader, val_loader, progress)
trained = handle_train_run(save_trained_run=save, cmd=cmd).unwrap()

history = pd.DataFrame(progress.rows, columns=["epoch", "train_loss", "val_loss"])
history.tail()

In [ ]:
plt.figure(figsize=(6, 3.5))
plt.plot(history["epoch"], history["train_loss"], label="train")
plt.plot(history["epoch"], history["val_loss"], label="val")
plt.xlabel("epoch")
plt.ylabel("MSE (standardized)")
plt.title("train vs val loss")
plt.legend()
plt.tight_layout()
plt.show()

## 5. Load + materialize, then report val + test metrics

We recover the model from disk (`load_trained_run` -> `materialize_model`, both reload
guards passing) and score it on the held-out **val** and **test** loaders, de-standardized
back to solar masses. Val and test RMSE being close is the generalization signal.

In [ ]:
def rmse_msun(model: nn.Module, dl: DataLoader) -> float:
    '''Root-mean-square error in solar masses over a loader.'''
    model.eval()
    se, n = 0.0, 0
    with torch.no_grad():
        for xb, yb in dl:
            pred = model(xb) * y_std + y_mean
            truth = yb * y_std + y_mean
            se += float(((pred - truth) ** 2).sum())
            n += yb.shape[0]
    return (se / n) ** 0.5


loaded = load_trained_run(CKPT_DIR, "ns-split").unwrap()
live = materialize_model(lambda: build_model(loaded.config), IDENTITY, CKPT_DIR, loaded).unwrap()
print("val  RMSE (Msun):", round(rmse_msun(live, val_loader), 4))
print("test RMSE (Msun):", round(rmse_msun(live, test_loader), 4))

## Findings + harvest candidates (L3g.1 -- flagged, not yet promoted)

**Structural finding.** Injecting pre-split train/val/test loaders required **no `src`
change**: the loaders ride in the injected adapter's closure and the application port
`SaveTrainedRunFn(ConfiguredRun) -> Result[Checkpoint]` is agnostic to loader count. So
the mlmodels end-goal "inject a NN + pre-split loaders" is already satisfiable by hand
today; what the API can add is *ergonomics* and *recording*, not new capability.

**Candidates to harvest into a BC** (let each stabilize across another spike first):

| Candidate | What | Target BC | Why general |
|---|---|---|---|
| split helper | seeded `perm` + fractions -> `(idx_train, idx_val, idx_test)` loaders | `datasets` or shared | pure, dataset-agnostic; pairs with `make_holdout_spec` |
| record the split | persist `HoldoutSpec` (fractions+seed) in the manifest | `mlmodels` | makes a run's split reproducible from disk (old L3a.5 aggregate ext.) |
| record metrics | a `Metrics` VO (val/test RMSE) + manifest field | `mlmodels` | persisted metrics = comparable runs + feeds the feedback pillar (old L3a.4) |
| tensor-ization | `df` + feature/target cols -> standardized tensors (+ the standardizer) | `datasets` or shared | recurs in every spike; the standardizer must travel with the model to predict |

**Next pillars:** L3f.2 continuous progress feedback (the `_Progress` callback, made
first-class), L3f.3 checkpoints + resume (where the fork-2 `trainer.save_checkpoint` +
the `materialize_model` format fix from `ns_mlmodels-e2e.ipynb` 8b land together).